# DNNet — Interactive Testing Notebook
Load your trained model and explore embeddings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dnnet_helper import DNNetHelper

## 1. Load the model

In [ ]:
CHECKPOINT = 'checkpoints/fold_0/best_model.pth'  # <- change this

dn = DNNetHelper(CHECKPOINT)
# Model ready on cpu  |  embedding dim = 128

## 2. Embed a single image

In [ ]:
emb = dn.embed_image('path/to/nose.jpg')

print('Shape :', emb.shape)       # (128,)
print('Norm  :', np.linalg.norm(emb))  # ~1.0 (L2-normalised)
print('Values:', emb[:8])         # first 8 dims

## 3. Embed a PIL image (useful if you already loaded it)

In [ ]:
img = Image.open('path/to/nose.jpg').convert('RGB')
emb = dn.embed_pil(img)
print(emb.shape)  # (128,)

## 4. Embed a list of images

In [ ]:
paths = [
    'dataset/dog_001/img_01.jpg',
    'dataset/dog_001/img_02.jpg',
    'dataset/dog_002/img_01.jpg',
]

embs = dn.embed_images(paths)
print('Shape:', embs.shape)   # (3, 128)

## 5. Embed every image in a folder

In [ ]:
embs, paths = dn.embed_folder('dataset/dog_001/')

print('Embeddings:', embs.shape)   # (N, 128)
print('Paths:', paths)

## 6. Similarity / distance between two images

In [ ]:
# Cosine similarity: closer to 1.0 = more similar
sim = dn.similarity('dataset/dog_001/img_01.jpg',
                    'dataset/dog_001/img_02.jpg')
print(f'Same dog similarity : {sim:.4f}')

sim2 = dn.similarity('dataset/dog_001/img_01.jpg',
                     'dataset/dog_002/img_01.jpg')
print(f'Diff dog similarity : {sim2:.4f}')

# Euclidean distance: lower = more similar
dist = dn.distance('dataset/dog_001/img_01.jpg',
                   'dataset/dog_002/img_01.jpg')
print(f'Euclidean distance  : {dist:.4f}')

## 7. Find closest match in a gallery

In [ ]:
results = dn.find_closest(
    query_path   = 'query_nose.jpg',
    gallery_path = './dataset/dog_001/',
    top_k        = 3,
)

for r in results:
    print(f"Rank {r['rank']}  dist={r['distance']:.4f}  sim={r['similarity']:.4f}  {r['path']}")

## 8. Visualise embeddings with t-SNE

In [ ]:
from sklearn.manifold import TSNE
import glob

# Collect images from multiple dogs
all_paths, all_labels, all_names = [], [], []
for label, dog_dir in enumerate(sorted(glob.glob('dataset/dog_*'))):
    imgs = sorted(glob.glob(f'{dog_dir}/*.jpg'))
    all_paths  += imgs
    all_labels += [label] * len(imgs)
    all_names.append(dog_dir.split('/')[-1])

embs = dn.embed_images(all_paths)
labels = np.array(all_labels)

perplexity = min(30, len(embs) - 1)
proj = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(embs)

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab20
for i, name in enumerate(all_names):
    mask = labels == i
    ax.scatter(proj[mask, 0], proj[mask, 1], label=name,
               c=[cmap(i % 20)], s=40, alpha=0.8)
ax.legend(fontsize=7, ncol=2)
ax.set_title('t-SNE of dog nose-print embeddings')
ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Pairwise distance heatmap

In [ ]:
# Show pairwise distances for a small set of images
sample_paths = all_paths[:20]
sample_labels = labels[:20]
sample_embs = dn.embed_images(sample_paths)

# Euclidean distance matrix
sq = np.sum(sample_embs**2, axis=1, keepdims=True)
dist_mat = np.sqrt(np.clip(sq + sq.T - 2 * sample_embs @ sample_embs.T, 0, None))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(dist_mat, cmap='viridis_r')
plt.colorbar(im, ax=ax, label='Euclidean distance')
ax.set_title('Pairwise distance matrix (darker = more similar)')
ax.set_xlabel('Image index')
ax.set_ylabel('Image index')
plt.tight_layout()
plt.show()

## 10. Save and reload embeddings

In [ ]:
# Save
np.savez('my_embeddings.npz',
         embeddings = embs,
         labels     = labels,
         paths      = np.array(all_paths))

# Reload later — no model needed
data   = np.load('my_embeddings.npz', allow_pickle=True)
embs   = data['embeddings']   # (N, 128)
labels = data['labels']       # (N,)
paths  = data['paths']        # (N,)
print(embs.shape)